In [1]:
!pip install --upgrade pandas sqlalchemy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from sqlalchemy import create_engine, types
import pyodbc

In [4]:
df = pd.read_csv('netflix_titles.csv')
print(df.head(2))

  show_id     type                 title         director  \
0      s1    Movie  Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show         Blood & Water              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   

                                         listed_in  \
0                                    Documentaries   
1  International TV Shows, TV Dramas, TV Mysteries   

                                         description  
0  As her father nears the end of his life, filmm...  
1  After crossing paths at a party, a Cape Town t...  


In [5]:
pyodbc.drivers()

['SQL Server',
 'ODBC Driver 17 for SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'ODBC Driver 18 for SQL Server']

In [6]:
engine = create_engine('mssql://Yassine/netflix_db?driver=ODBC+Driver+17+for+SQL+Server')
conn = engine.connect()

In [7]:
dtype_map = {
    "show_id":      types.NVARCHAR(10),
    "type":         types.NVARCHAR(10),
    "title":        types.NVARCHAR(200),
    "director":     types.NVARCHAR(250),
    "cast":         types.NVARCHAR(1000),
    "country":      types.NVARCHAR(150),
    "date_added":   types.NVARCHAR(20),
    "rating":       types.NVARCHAR(10),
    "duration":     types.NVARCHAR(10),
    "listed_in":    types.NVARCHAR(100),
    "description":  types.NVARCHAR(500),
}

In [8]:
df.to_sql(
    "netflix_raw",
    con=conn,
    index=False,
    if_exists="replace",
    dtype=dtype_map
)

conn.close()

In [20]:
df[df.show_id == 's5023']

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5022,s5023,Movie,반드시 잡는다,Hong-seon Kim,Baek Yoon-sik,South Korea,"February 28, 2018",2017,TV-MA,110 min,"Dramas, International Movies, Thrillers",After people in his town start turning up dead...


In [17]:
for champ in df.columns:
    if df[champ].dtype == 'string':
        max_len = max(df[champ].str.len())
    else:
        max_len = max(df[champ].apply(lambda x: len(str(x))))
    print(f"Champ: {champ}, Longueur max: {max_len}")

Champ: show_id, Longueur max: 5
Champ: type, Longueur max: 7
Champ: title, Longueur max: 104
Champ: director, Longueur max: 208
Champ: cast, Longueur max: 771
Champ: country, Longueur max: 123
Champ: date_added, Longueur max: 19
Champ: release_year, Longueur max: 4
Champ: rating, Longueur max: 8
Champ: duration, Longueur max: 10
Champ: listed_in, Longueur max: 79
Champ: description, Longueur max: 248
